# Packing Data for SeisBench (Machine Learning)

Welcome to the Grand Finale of our dataset curation!

Right now, we have hundreds of tiny `.mseed` files and `.json` files scattered in a folder. While this is great for us to read, Machine Learning models (like Neural Networks) hate opening thousands of tiny files—it's incredibly slow.

Instead, we use a library called **SeisBench**. SeisBench expects us to pack all our waveforms into one massive, super-fast container called an **HDF5** file (think of it like a highly optimized ZIP file for science). It also expects a single `metadata.csv` spreadsheet containing all the labels (our JSON answer keys).

Let's write a script to pack our data into this official SeisBench format!

### Step 1: Loading Libraries and Setup
We import `seisbench.data` as `sbd`. This is the tool that will do the heavy lifting of packing our container.

In [6]:
import numpy as np
import json
import glob
import os
from obspy import read
import seisbench.data as sbd

input_dir = 'trimmed_and_consistent_mseed'
output_dir = 'final_curated_seisbench_data'

os.makedirs(output_dir, exist_ok=True)

### Step 2: The Machine Learning Split (Train, Dev, Test)
Before we train an AI, we have to split our data into three piles:
1. **Train (70%)**: The "Homework" the AI uses to learn.
2. **Dev (15%)**: The "Practice Test" the AI uses to tune its strategies.
3. **Test (15%)**: The "Final Exam" the AI takes at the very end to prove it actually learned!

We write a quick function to randomly assign each of our events into one of these three piles.

In [7]:
def assign_dataset_splits(n_events, split_ratios={'train': 0.7, 'dev': 0.15, 'test': 0.15}, random_seed=42):
    np.random.seed(random_seed) # Seed ensures we get the same random split every time
    
    n_train = int(np.floor(n_events * split_ratios['train']))
    n_dev = int(np.floor(n_events * split_ratios['dev']))
    n_test = n_events - n_train - n_dev
    
    splits = ['train'] * n_train + ['dev'] * n_dev + ['test'] * n_test
    np.random.shuffle(splits) # Shuffle the deck!
    
    return splits

### Step 3: Organizing the Channels (Z, N, E)
AI Models are very picky. When they look at a 3-component seismogram, they expect the data in certain order: 
1. **Z** (Up-Down)
2. **N** (North-South)
3. **E** (East-West)

Because our raw files are sometimes named `HH1` or `HH2`, we need a helper function to translate them and ensure we stack our data in the strict `Z, N, E` order.

In [8]:
def map_channel_to_component(chan):
    if chan.endswith('Z'): return 'Z'              # Vertical
    if chan.endswith('N') or chan.endswith('2'): return 'N' # North or Horizontal 2
    if chan.endswith('E') or chan.endswith('1'): return 'E' # East or Horizontal 1
    return None

### Step 4: Packing the Container
Now we do the heavy lifting! 
1. We find all our JSON files.
2. We open the `WaveformDataWriter` which handles building the `.hdf5` and `.csv` files.
3. We loop through every JSON file, read its associated waveforms, stack them into `Z, N, E` order, and hand them to the writer!

In [9]:
json_files = sorted(glob.glob(os.path.join(input_dir, '*.json')))
print(f"Found {len(json_files)} metadata JSON files to pack.")

# Randomly assign Train/Dev/Test to our events
splits = assign_dataset_splits(len(json_files))

metadata_path = os.path.join(output_dir, 'metadata.csv')
waveforms_path = os.path.join(output_dir, 'waveforms.hdf5')

# Open the SeisBench container writer
with sbd.WaveformDataWriter(metadata_path, waveforms_path) as writer:
    # Tell SeisBench exactly what shape our data is in
    writer.data_format = {
        'dimension_order': 'CW',  # C = Channels (3), W = Width/Samples (2001)
        'component_order': 'ZNE', # Vertical, North, East
        'measurement': 'velocity',
        'unit': 'counts',
    }
    
    traces_written = 0
    
    for i, json_path in enumerate(json_files):
        with open(json_path, 'r') as f:
            metadata = json.load(f)
            
        # Create an empty dictionary to hold our 3 channels temporarily
        data_dict = {}
        
        # Load the waveforms listed in the JSON file
        for mseed_file, channel in zip(metadata['files']['mseed'], metadata['channels']):
            full_path = os.path.join(input_dir, mseed_file)
            st = read(full_path)
            comp = map_channel_to_component(channel)
            data_dict[comp] = st[0].data
                
        # Stack them perfectly into Z, N, E order! (Shape: 3 rows, 2001 columns)
        data_3c = np.vstack([data_dict['Z'], data_dict['N'], data_dict['E']])
        
        # Translate our JSON dictionary into SeisBench's strict naming format
        trace_metadata = {
            'trace_name_original': f"{metadata['network']}.{metadata['station']}.{metadata['start_time']}",
            'station_network_code': metadata['network'],
            'station_code': metadata['station'],
            'trace_channel': metadata['channels'][0][:2],
            'trace_sampling_rate_hz': metadata['sample_rate'],
            'trace_npts': data_3c.shape[1],
            'trace_start_time': metadata['start_time'],
            'source_id': metadata['event_id'],
            'split': splits[i] # Assign it to Train, Dev, or Test
        }
        
        # Add the P and S wave answers
        if 'p_arrival_sample' in metadata:
            trace_metadata['trace_p_arrival_sample'] = metadata['p_arrival_sample']
            trace_metadata['trace_p_status'] = 'manual'
            
        if 's_arrival_sample' in metadata:
            trace_metadata['trace_s_arrival_sample'] = metadata['s_arrival_sample']
            trace_metadata['trace_s_status'] = 'manual'
            
        # Pack it into the container!
        writer.add_trace(trace_metadata, data_3c)
        traces_written += 1

print(f"\n✓ Successfully packed {traces_written} traces into the SeisBench HDF5 container!")

Found 370 metadata JSON files to pack.


Traces converted: 370it [00:00, 1392.75it/s]


✓ Successfully packed 370 traces into the SeisBench HDF5 container!


### Step 5: Verify the Final Product
Let's test our new super-fast container by loading it back up using SeisBench!

In [10]:
dataset = sbd.WaveformDataset(output_dir)
print(f"Loaded dataset with {len(dataset)} traces!")

# Let's look at the Homework vs Final Exam splits
df = dataset.metadata
print("\nDataset splits:")
for split_name in ['train', 'dev', 'test']:
    count = (df['split'] == split_name).sum()
    print(f"  {split_name.capitalize()}: {count} traces")

2026-04-24 21:31:13,561 | seisbench | WARNING | Component order not specified, defaulting to 'ZNE'.


Loaded dataset with 370 traces!

Dataset splits:
  Train: 259 traces
  Dev: 55 traces
  Test: 56 traces


---
**You Did It!**
You have successfully built an automated, production-grade seismology dataset! 
Your waveforms are properly dimensioned (Z, N, E), randomly split for unbiased training, and neatly packed into a lightning-fast HDF5 file. You are fully ready to train AI models to detect icequakes!